# Reasoning-language experiment: Kaggle runner

Runs `src/run.py` from the repo on a free Kaggle GPU. Two stages:

1. **Compliance check** (2 questions x 3 languages x 3 prompt variants, ~18 traces, a few minutes): does Qwen3-1.7B follow "reason in X" at all?
2. **Main run** (250 questions x 4 languages x 2 arms = 2000 traces, roughly 1-2 h with vLLM).

**Notebook settings (right-hand panel):** Accelerator = GPU T4 x1 (or P100). Internet = ON (needed to download the model and pip packages). Persistence = Files only is fine.

Results are written to `/kaggle/working/results/`. Rows are appended as they finish and the run resumes if restarted, so an interrupted session is not lost. For the main run, prefer **Save Version -> Save & Run All (Commit)** so it runs headless for up to 12 h; the files then appear under the notebook's *Output* tab.

In [ ]:
# 1. Get the code from GitHub.
REPO_URL = "https://github.com/tomf0/llm_mini_replication.git"

import os, subprocess
os.chdir("/kaggle/working")
if not os.path.isdir("repo"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "repo"], check=True)
else:
    subprocess.run(["git", "-C", "repo", "pull"], check=True)
os.chdir("/kaggle/working/repo")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
!ls

In [ ]:
# 2. Dependencies. vLLM is large and may take several minutes; it pulls its own torch build.
#    If this cell fails, or vLLM crashes at load time in a later cell, set BACKEND = "hf" below and rerun.
!pip install -q lingua-language-detector datasets pyyaml
!pip install -q vllm
# vLLM brings its own torch build; Kaggle's preinstalled torchaudio was built against a different CUDA
# and makes `transformers` crash on import. We never use audio, so remove it.
!pip uninstall -y -q torchaudio

import importlib
BACKEND = "vllm" if importlib.util.find_spec("vllm") else "hf"
print("backend:", BACKEND)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3. Compliance check on the real model (small, fast). Read the examples: is the reasoning actually in French/Chinese/Swahili?
EXTRA = "" if BACKEND == "vllm" else "--batch-size 8"
!python src/run.py --config config/kaggle_compliance_check.yaml --backend {BACKEND} {EXTRA} \
    --output /kaggle/working/results/compliance_check.jsonl
!python analysis/quicklook.py /kaggle/working/results/compliance_check.jsonl --examples 2

### Decision point
If the `instruct_*` rows show `comply = 0` for every language and `prefill` is the only variant that works, that is itself a finding: the main run below includes both arms (`instruct_en` and `prefill`) so compliance is reported per arm rather than assumed. Nothing needs changing. Continue.

In [ ]:
# 4. Main run: 2000 traces. Resumable: rerunning this cell skips rows already in the file.
EXTRA = "" if BACKEND == "vllm" else "--batch-size 16"
!python src/run.py --config config/kaggle_main.yaml --backend {BACKEND} {EXTRA} \
    --output /kaggle/working/results/main_qwen3_1.7b_seed0.jsonl

In [ ]:
# 5. Summary + zip for download (also visible under the notebook's Output tab after a commit).
!python analysis/quicklook.py /kaggle/working/results/main_qwen3_1.7b_seed0.jsonl
!cd /kaggle/working && zip -q -r results.zip results && ls -lh results.zip results/